# Final Project: Network Door Security System (PYNQ #1)

Device Roles

PYNQ #1: Controls the alarm function of the door security system. Will emit a loud buzzing sound and flash a RGB board bright red when sound is detected with a sound sensor. This board will automatically be listening for the sound sensor board when the code is run.

PYNQ #2: Controls the sound sensor board. When a sound is detected it will send a signal to PYNQ board #2 and buzz the buzzer and flash the LED.

Wiring to this board:

Buzzer Module Wiring to PYNQ PMODA
- (-) pin connected to GND 
- (+) pin connectted to PMOD PIN2
- Middle pin is not connected

RGB LED Connected to PYNQ PMODB
- (-) pin connected to GND
- R pin connected to Pin 3
- G pin connected to Pin 2
- B pin connected to Pin 1

In [57]:
from multiprocessing import Process
import threading
import time
from pynq.overlays.base import BaseOverlay
base = BaseOverlay("base.bit")
import socket
import sys
import os

btns = base.btns_gpio
stop_alarm = False

In [58]:
%%microblaze base.PMODA
// Buzzer Controls
#include "gpio.h"
#include "pyprintf.h"
#include <unistd.h>

static int inited = 0;
static gpio pins[8];

void init_pmoda()
{
    if(inited) return;

    for(int i = 0; i < 8; i++)
    { 
        pins[i] = gpio_open(i); 
        gpio_set_direction(pins[i], GPIO_OUT); 
        gpio_write(pins[i], 0); 
    } 

    inited = 1;   
}

void write_gpio(unsigned int pin, unsigned int val)
{ 
    if (!inited) init_pmoda(); 
    if (pin >= 8) return;
    gpio_write(pins[pin], val);
}

void reset_all()
{
    if (!inited) init_pmoda();
    for(int i = 0; i < 8; i++)
        gpio_write(pins[i], 0);
}

// Buzzer
void buzz(unsigned int pin,
          unsigned int freq_hz,
          unsigned int duration_ms)
{
    if (!inited) init_pmoda();

    if (pin >= 8) { pyprintf("pin must be 0-7\n"); return; }
    if (freq_hz == 0) { pyprintf("freq must be > 0\n"); return; }

    // 50% duty cycle
    unsigned int period_us = 1000000u / freq_hz;
    if (period_us == 0) period_us = 1;

    unsigned int half_period = period_us / 2;
    if (half_period == 0) half_period = 1;

    unsigned int cycles = (duration_ms * 1000u) / period_us;

    for (unsigned int i = 0; i < cycles; i++)
    {
        // write value of 1
        gpio_write(pins[pin], 1);
        
        //sleep for 1/(2*tone_freq)
        usleep((useconds_t)half_period);

        // write value of 0
        gpio_write(pins[pin], 0);
        
        //sleep for 1/(2*tone_freq)
        usleep((useconds_t)half_period);
    }

    gpio_write(pins[pin], 0);  // ensure off
}

void high_beep(unsigned int pin)
{
    buzz(pin, 3500, 300);
}

In [59]:
%%microblaze base.PMODB
// RGB LED Controls
#include "gpio.h"
#include "pyprintf.h"
#include <unistd.h>
static int inited = 0;
static gpio pins[8];
void init_pmoda()
{
    if(inited)
        return;
    for(int i = 0; i < 8; i++)
    { 
        pins[i] = gpio_open(i); 
        gpio_set_direction(pins[i], GPIO_OUT); 
        gpio_write(pins[i], 0); 
    } 
    inited = 1;   

}
//Function to turn on/off a selected pin of PMODA
void write_gpio(unsigned int pin, unsigned int val){ 
    if (val > 1){ 
        pyprintf("pin value must be 0 or 1"); 
    } 
    if(!inited) 
    { 
        init_pmoda(); 
    } 
    gpio_write(pins[pin], val);
}

//Function to read the value of a selected pin of PMODA
unsigned int read_gpio(unsigned int pin){ 
    gpio pin_in = gpio_open(pin); 
    gpio_set_direction(pin_in, GPIO_IN); 
    return gpio_read(pin_in);
}

// reset GPIO bins meaning writing 0 as output to all pins 0 - 7
void reset_pin(unsigned int pin)
{ 
    if(!inited) 
    { 
        init_pmoda(); 
    } 
    write_gpio(pin, 0); 
    //read_gpio(pin);
}

void run_pwm(unsigned int pin, 
             unsigned int freq_hz,
             unsigned int duty_milli, 
             unsigned int duration_ms)
{ 
    if (!inited) init_pmoda(); 
    
    if (pin >= 8) { pyprintf("pin must be 0-7\n"); return; } 
    if (freq_hz == 0) { pyprintf("freq_hz must be > 0\n"); return; } 
    if (duty_milli > 1000) duty_milli = 1000; 
    
    // corner cases: 0% / 100% 
    if (duty_milli == 0) { 
        gpio_write(pins[pin], 0); 
        usleep((useconds_t)duration_ms * 1000); 
        return; 
    } 
    
    if (duty_milli >= 1000) {
        gpio_write(pins[pin], 1); 
        usleep((useconds_t)duration_ms * 1000);
        // turn LED off
        gpio_write(pins[pin], 0);   
        return;
    
    }

    // period in microseconds
    unsigned int period_us = 1000000u / freq_hz; // how many us per cycle
    if (period_us == 0) period_us = 1; // avoid 0 due to rounding ---- safety
    
    unsigned int on_us = (period_us * duty_milli) / 1000u;
    unsigned int off_us = period_us - on_us;

    pyprintf("on_us is %ui", on_us);
    pyprintf("on_us is %ui", off_us);

    // avoid 0 due to rounding ---- safety
    if (on_us == 0) on_us = 1;
    if (off_us == 0) off_us = 1;

    unsigned int cycles = (duration_ms * 1000u) / period_us;

    for (unsigned int i = 0; i < cycles + 1; i++) {
        gpio_write(pins[pin], 1);
        usleep((useconds_t)on_us);
        gpio_write(pins[pin], 0);
        usleep((useconds_t)off_us);
    }
}

void run_pwm_for6(unsigned int pin)
{
    run_pwm(pin, 100, 250, 1000);
    write_gpio(pin, 0);
    usleep((useconds_t)1000000u);
}

In [50]:
# Code that runs LED from HW1 to Test RGB LED
for pin_num in range(8): 
    reset_pin(pin_num);
#write_gpio(1, 1)

# run_pwm(pin, frequency, duty cycle, duration)

# Turns on red LED and turns off after duration is over
run_pwm(3, 10, 988, 750) 

In [23]:
# Test code that beeps the buzzer for 750 ms
buzz(2, 4500, 750)

In [24]:
# Test Code for alarm loop. Will buzz the buzzer and turn on the LED at the same time and run indefinitely until turn off

while True:
    
    # Define thread of led and command to run LED
    # writes to pin 3 and turns on RED RGB LED for 750 ms at a frequency of 75Hz and 100% LED Brightness
    thread_led = threading.Thread(
        target=run_pwm,
        args=(3, 75, 1000, 750)
    )

    # starts thread
    thread_led.start()

    # runs the buzzer at a frequency of 4.5 KHz for 750 ms
    buzz(2, 4500, 750)

    # stops thread
    thread_led.join()

    # Pauses thread for 1 second
    time.sleep(1)

KeyboardInterrupt: 

In [84]:
# Function that controls the alarm functionality (RBG LED + Buzzer execution)
def alarm():

    # glohal variable that keeps track if alarm is activated
    global alarm_active

    # while loop for when the alarm is not stopped
    while not stop_alarm:

        # if the alarm is active
        if alarm_active:

            # run buzzer + LED briefly
            buzz(2, 4500, 650)
            run_pwm(3, 75, 1000, 650)

            # combines the 0.25s duration of the alarm/led active together
            # loop will run 4 times for combined 1 second duration
            for _ in range(4):
                
                if not alarm_active:
                    
                    # Resets all PMOD A/B pins
                    reset_all()
                    # breaks out of loop
                    break
                
                # pauses thread for 0.25 seconds
                time.sleep(0.25)

        # when the alarm is not active
        else:
            # pauses thread for 0.02 seconds
            time.sleep(0.02)

# Code that controls the server side of the PYNQ 1 Board
def server():

    # glohal variable that keeps track if alarm is activated
    global alarm_active
    
    # Defines host IP address and port
    HOST = ''
    PORT = 50007

    # creating a socket
    s_server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    # allows for reusing the same address. helps prevent same address errors
    s_server.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    print('Socket created')
    
    # 1: Bind the socket to the pynq board <CLIENT-IP> at port <LISTENING-PORT>
    s_server.bind((HOST, PORT))
    print('Socket bind complete')
    
    # listens for connections
    s_server.listen(10)

    # 2: Accept connections
    print('Waiting for sound sensor board to connect...')
    conn, addr = s_server.accept()
    # sets the socket in non-blocking mode, so that buttons work bi-directionally
    conn.setblocking(False)
    
    # server starts with the alarm feature off and disarmed
    alarm_active = False
    armed = False
    print("\nBUTTON OPTIONS:")
    print("Press Button 1 to arm security system...")
    print("Press Button 3 to disconnect...")
    
    print("\n------------------------------------------------")
    print('Connected by', addr)

    # alarm thread declaration
    alarm_thread = threading.Thread(target=alarm, daemon=True)
    # start alarm thread
    alarm_thread.start()
    
    # Main while loop for server listening for button presses
    while True:

        # Button 1: Case where the security system is armed with Button 1 on PYNQ1
        if base.buttons[1].read() == 1:
            print("\n------------------------------------------------")
            print("Button 1 Pressed - Security system armed")
            print("Listening for intruders...")
            
            print("\nBUTTON OPTIONS:")
            print("Press Button 2 to disarm security system...")
            print("Press Button 3 to disconnect from server...")
            
            # arms sound sensor
            armed = True
            # Sends sound sensor arming message
            conn.sendall(b'ARM')
            time.sleep(0.3)

        # Button 2 Case: Disarm when the alarm board is actively buzzing/flashing with Button 2 on PYNQ1
        if base.buttons[2].read() == 1:
            print("\n------------------------------------------------")
            print("Button 2 Pressed - Security system disarmed")
            
            print("\nBUTTON OPTIONS:")
            print("Press Button 1 to arm security system...")
            print("Press Button 3 to disconnect from server...")
            
            # turns the alarm off
            alarm_active = False
            # disarms sound sensor
            armed = False
            # Resets all PMOD A/B pins
            reset_all()
            # Sends sound sensor disarming message
            conn.sendall(b'DISARM')
            time.sleep(0.3)

        # Button 3 Case: Disconnect when the alarm board is actively buzzing/flashing with Button 3 on PYNQ1
        if base.buttons[3].read() == 1:
            print("\n------------------------------------------------")
            print("Button 3 Pressed - Alarm board disconnected from client")
            
            # Sends sound sensor disconnect message
            conn.sendall(b'DISCONNECT')   
            # alarm is turned off
            stop_alarm = True
            
            # closes socket and the connection after disconnect
            conn.close()
            s_server.close()
            return

        try:
            # received button data at full bytes from PYNQ #2 Board
            pynq2_data = conn.recv(1024)

            # Button 1: Case where the security system is armed with Button 1 on PYNQ2
            if pynq2_data == b'ARM':
                print("\n------------------------------------------------")
                print("Button 1 Pressed - Security system armed")
                print("Listening for intruders...")

                print("\nBUTTON OPTIONS:")
                print("Press Button 2 to disarm security system...")
                print("Press Button 3 to disconnect from server...")
                
                # arms sound sensor
                armed = True
                # Sends sound sensor arming message
                conn.sendall(b'ARM')

            # Button 2 Case: Disarm when the alarm board is actively buzzing/flashing with Button 2 on PYNQ2
            elif pynq2_data == b'DISARM':
                print("\n------------------------------------------------")
                print("Button 2 Pressed - Security system disarmed")

                print("\nBUTTON OPTIONS:")
                print("Press Button 1 to arm security system...")
                print("Press Button 3 to disconnect from server...")
                
                # turns the alarm off
                alarm_active = False
                # disarms sound sensor
                armed = False
                # Sends sound sensor disarming message
                conn.sendall(b'DISARM')
                # Resets all PMOD A/B pins
                reset_all()

            # Alarm running case: alarm will run if alarm message is received AND the sound sensor is armed
            elif pynq2_data == b'ALARM':
                if armed:
                    print("\n------------------------------------------------")
                    print("UNAUTHORIZED ACCESS DETECTED")

                    print("\nBUTTON OPTIONS:")
                    print("Press Button 2 to disarm security system...")
                    print("Press Button 3 to disconnect from server...")
                    
                    # turns the alarm on
                    alarm_active = True

            # Button 3 Case: Disconnect when the alarm board is actively buzzing/flashing with Button 3 on PYNQ2
            elif pynq2_data == b'DISCONNECT':
                print("\n------------------------------------------------")
                print("Button 3 Pressed - Sound sensor board disconnected from server")
                
                # Sends sound sensor disconnect message
                conn.sendall(b'DISCONNECT')
                # alarm is turned off
                stop_alarm = True
                return

        # ignores potential blocks 
        except BlockingIOError:
            pass
        
            # # run alarm until button 2 is pressed on PYNQ 2 board
            # if alarm_active:

                # thread_buzzer = threading.Thread(
                #        target=buzz,
                #        args=(2,4500,650)
                #    )

                # # Thread that runs the RGB LED board indefinitely until it is disarmed
                # thread_led = threading.Thread(
                #    target=run_pwm,
                #    args=(3,75,1000,650)
                # )

                # # Runs buzzer and led
                # thread_buzzer.start()
                # thread_led.start()

        time.sleep(0.02)
        
    # closes socket after the loop
    s_server.close()
    print("Server socket closed")

In [85]:
# Server process turns on upon code execution

print("Starting Alarm server...")

try:
    # Server process definition
    p = Process(target=server)
    # Starts server process
    p.start()
    # time.sleep(1)

    # Button 0 Starts the Client
    # print("Press Button 0 to connect to sound sensor board...")
    # while base.buttons[0].read() == 0:
    #    time.sleep(0.1)

    # print("\n------------------------------------------------")
    # print("Button 0 Pressed")
    # print("starting client")
    
    # Completes server process
    p.join()
    print("Cell execution complete")
    
# Kill processes if button doesnt work (For debugging)
except KeyboardInterrupt:
    p.terminate()
    # print("Server process terminated")
    p.join()
    print("Cell execution complete")
    print('Interrupt')
    sys.exit(0)

Starting Alarm server...
Socket created
Socket bind complete
Waiting for sound sensor board to connect...

BUTTON OPTIONS:
Press Button 1 to arm security system...
Press Button 3 to disconnect...

------------------------------------------------
Connected by ('192.168.0.67', 57198)

------------------------------------------------
Button 1 Pressed - Security system armed
Listening for intruders...

BUTTON OPTIONS:
Press Button 2 to disarm security system...
Press Button 3 to disconnect from server...

------------------------------------------------
UNAUTHORIZED ACCESS DETECTED

BUTTON OPTIONS:
Press Button 2 to disarm security system...
Press Button 3 to disconnect from server...

------------------------------------------------
Button 2 Pressed - Security system disarmed

BUTTON OPTIONS:
Press Button 1 to arm security system...
Press Button 3 to disconnect from server...

------------------------------------------------
Button 1 Pressed - Security system armed
Listening for intruders